# HUST
Compute specimen outcomes using the fixed analysis settings. G4 is the state-variance warning rule. The two batches use separate settings and are pooled for reporting. The prepared capacities supply the inputs to the recursive state estimate.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
code_dir = next(
    (candidate for folder in (cwd, *cwd.parents)
     for candidate in (folder, folder / "submission_code")
     if (candidate / "manuscript_replay.py").is_file()
     and (candidate.parent / "submission_inputs").is_dir()),
    None,
)
if code_dir is None:
    raise FileNotFoundError("Open the project containing submission_code and submission_inputs.")
sys.path.insert(0, str(code_dir))

In [ ]:
import pandas as pd
from battery_models import CONFIGS
from manuscript_replay import settings

batches = ['HUST1', 'HUST2']
parameters = []
for batch in batches:
    detector, stage, thresholds = settings(batch)
    parameters.append(dict(
        batch=batch, variance_window=detector["w"], smoothing=detector["m"],
        evidence_window=detector["d"], threshold=thresholds["G4"],
        rls_lambda=CONFIGS[batch].rls_lambda, rls_delta=CONFIGS[batch].rls_delta,
        stage_duration=stage["duration_points"],
    ))
pd.DataFrame(parameters)

In [ ]:
from release_replay import reproduce, summarize

outcomes = reproduce(batches)
summarize(outcomes)

In [ ]:
outcomes.query("role == 'evaluation_60pct' and method == 'G4'")[
    ["id", "transition_s", "alarm_s", "lead", "warning", "at_transition", "science_valid"]
]